# Brazilian E-Commerce Data Preparation

## Project Overview

This notebook focuses on data preparation and feature engineering using the Brazilian E-Commerce Public Dataset by Olist.

The objective is to clean, transform, and combine multiple raw datasets into a single master dataset that can be used for further analysis and interactive dashboard development in Power BI.

### Key Objectives

- Explore the structure and quality of the raw datasets.
- Identify and handle duplicate and missing records.
- Standardize data formats and categorical values.
- Combine order, customer, product, seller, payment, and review information.
- Create analytical features related to revenue, delivery performance, customer reviews, and payment behavior.
- Generate a consolidated dataset for business intelligence reporting.

### Final Output

A consolidated order-level dataset exported as `Master_ecommerce.csv`, intended for further analysis and Power BI visualization.

## 1. Importing Libraries

Importing the required Python libraries for data manipulation, numerical operation.

- **Pandas:** Data loading, cleaning, transformation, and aggregation.
- **NumPy:** Numerical operations and conditional feature creation.

In [1]:
import pandas as pd
import numpy as np

## 2. Loading the Datasets

The Olist dataset consists of multiple CSV files containing information about different aspects of the e-commerce business.

The following datasets are loaded:

- Orders
- Customers
- Order Items
- Payments
- Reviews
- Products
- Sellers
- Product Category Name Translation

These datasets will be explored and transformed before being combined into a consolidated master dataset.

In [2]:
raw = '../Raw_data/'
orders = pd.read_csv(raw + 'olist_orders_dataset.csv')
customers = pd.read_csv(raw + 'olist_customers_dataset.csv')
items = pd.read_csv(raw + 'olist_order_items_dataset.csv')
payments = pd.read_csv(raw + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(raw + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(raw + 'olist_products_dataset.csv')
sellers = pd.read_csv(raw + 'olist_sellers_dataset.csv')
eng_name = pd.read_csv(raw + 'product_category_name_translation.csv')

## 3. Initial Data Exploration

An initial examination of all datasets is performed to understand their structure and identify potential data quality issues.

The exploration includes:

- Column names and available attributes.
- Data types of individual columns.
- Dataset information and non-null counts.
- Identification of columns requiring type conversion or further cleaning.

This step helps establish the necessary data preparation tasks before combining the datasets.

In [3]:
da = {'Orders': orders,'Customers' : customers,'Items' : items,'Payments' : payments,
      'Reviews' : reviews,'Products' : products,'Sellers' : sellers,'Eng_name' : eng_name}
for name,df in da.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {name}")
    
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nInfo:")
    df.info()

    print("\nDataType")
    print(df.dtypes.to_frame('DataType'))    


Dataset: Orders

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

DataType
                           

## 4. Data Cleaning – Duplicate Records

Duplicate records are removed from the datasets to maintain data consistency and prevent unintended duplication during aggregation and merging.

For datasets containing unique identifiers, duplicate removal is performed using the relevant identifier columns.

For payment and review datasets, exact duplicate rows are removed.

In [4]:
orders = orders.drop_duplicates(subset='order_id')
customers = customers.drop_duplicates(subset='customer_id')
items = items.drop_duplicates(subset=['order_id', 'order_item_id'])
products = products.drop_duplicates(subset='product_id')
payments = payments.drop_duplicates()
reviews = reviews.drop_duplicates()
sellers = sellers.drop_duplicates()
eng_name = eng_name.drop_duplicates()

## 5. Data Type Conversion

Date and timestamp columns are converted into appropriate datetime formats using Pandas.

This transformation is necessary for:

- Calculating delivery durations.
- Measuring delivery delays.
- Analyzing monthly and yearly order trends.
- Creating time-based features.

Invalid or unparseable date values are converted to missing values for subsequent handling.

In [5]:
cols = ['order_purchase_timestamp','order_approved_at', 'order_delivered_carrier_date','order_delivered_customer_date', 'order_estimated_delivery_date']
for i in cols:
    orders[i] = pd.to_datetime(orders[i],errors='coerce')

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'],errors='coerce')

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")

reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

## 6. Standardizing Categorical Data

Text-based columns are standardized to improve consistency across datasets.

The cleaning process includes:

- Removing unnecessary leading and trailing spaces.
- Standardizing customer and seller city names.
- Converting state abbreviations to uppercase.
- Handling missing product category names.
- Standardizing translated product category names.

These transformations help improve grouping, filtering, and aggregation accuracy.

In [6]:
customers['customer_city'] = customers['customer_city'].str.strip().str.title()
customers['customer_state'] = customers['customer_state'].str.strip().str.upper()

sellers['seller_city'] = sellers['seller_city'].str.strip().str.title()
sellers['seller_state'] = sellers['seller_state'].str.strip().str.upper()

products['product_category_name'] = products['product_category_name'].fillna('Unknown')
eng_name['product_category_name_english'] = eng_name['product_category_name_english'].str.strip().str.lower()

## 7. Filtering Order Records

Orders with `canceled` and `unavailable` statuses are excluded from the working dataset.

This filtering is applied to focus the analysis on orders with usable transaction information.

The remaining order statuses are retained for further processing. Delivery-related calculations are evaluated separately because not all retained orders necessarily have completed deliveries.

In [7]:
orders = orders[~orders['order_status'].isin(['canceled', 'unavailable'])].copy()

## 8. Product Category Translation

The product category translation dataset is merged with the products dataset using the original Portuguese category name.

This provides English category labels, making the categories easier to understand and interpret in the final dashboard.

Unmatched category names are assigned an `Unknown` label.

In [8]:
products = products.merge(eng_name,on='product_category_name',how='left')

In [9]:
products['product_category_name_english'] = products['product_category_name_english'].fillna('Unknown')

## 9. Payment Data Aggregation

Payment information is aggregated at the order level because an order can contain multiple payment records.

The following features are created:

- Total payment value.
- Average payment installments.
- Maximum payment installments.
- Number of distinct payment types.
- Main payment type.

The main payment type is identified based on the most frequently occurring payment type within each order, with alphabetical ordering used to break ties.

In [10]:
payments_agg = payments.groupby('order_id',as_index=False).agg(
    total_payment_value = ('payment_value','sum'),
    avg_payment_installments = ('payment_installments','mean'),
    max_payment_installments = ('payment_installments','max'),
    payment_type_nunique = ('payment_type','nunique')
)

payment_mode = (
    payments.groupby(['order_id','payment_type']).size().reset_index(name='Count')
    .sort_values(['order_id','Count', 'payment_type'], ascending=[True, False, True])
    .drop_duplicates('order_id')[['order_id','payment_type']].rename(columns={'payment_type': 'main_payment_type'})
               )

payments_agg = payments_agg.merge(payment_mode, on='order_id', how='left')

## 10. Customer Review Aggregation

Review records are aggregated at the order level to create customer satisfaction metrics.

The following features are generated:

- Average review score per order.
- Number of review records per order.
- Latest review answer timestamp.

This aggregation ensures that review information can be incorporated into the order-level master dataset without introducing additional rows for multiple review records.

In [11]:
reviews = reviews.sort_values(['order_id', 'review_answer_timestamp'])

reviews_agg = reviews.groupby('order_id', as_index=False).agg(
    review_score=('review_score', 'mean'),
    review_count=('review_id', 'count'),
    latest_review_date=('review_answer_timestamp', 'max')
)

## 11. Item Data Enrichment

The order items dataset is enriched by merging product and seller information.

This combines:

- Product category details.
- English product category names.
- Seller city.
- Seller state.

The enriched dataset provides additional product and geographic information for subsequent order-level analysis.

In [12]:
items_enriched = (
    items
    .merge(products[['product_id', 'product_category_name', 'product_category_name_english']],
           on='product_id', how='left')
    .merge(sellers[['seller_id', 'seller_city', 'seller_state']],
           on='seller_id', how='left')
)

In [13]:
items_enriched['product_category_name'] = items_enriched['product_category_name'].fillna('unknown')
items_enriched['product_category_name_english'] = items_enriched['product_category_name_english'].fillna('unknown')

## 12. Order-Level Item Aggregation

Item-level information is aggregated by `order_id` to create order-level product and freight metrics.

The following features are generated:

- Total item value.
- Total freight value.
- Total number of items.
- Number of unique products.
- Number of unique sellers.

This aggregation ensures that multiple order items are summarized into a single record per order.

In [14]:
items_agg = items_enriched.groupby('order_id', as_index=False).agg(
    total_item_value=('price', 'sum'),
    total_freight_value=('freight_value', 'sum'),
    total_items=('order_item_id', 'count'),
    unique_products=('product_id', 'nunique'),
    unique_sellers=('seller_id', 'nunique')
)

## 13. Identifying Main Product Category and Seller

Since an order may contain products from multiple categories and sellers, a representative category and seller are identified for each order.

The main category is determined by the most frequently occurring product category within an order.

Similarly, the main seller is identified based on the highest number of items associated with that seller.

Where multiple categories or sellers have the same frequency, alphabetical ordering is used to determine the representative value.

These features allow order-level segmentation by category and seller, while the aggregated item metrics retain the overall order information.

In [15]:
category_mode = (
    items_enriched.groupby(['order_id', 'product_category_name_english'])
    .size()
    .reset_index(name='cnt')
    .sort_values(['order_id', 'cnt', 'product_category_name_english'], ascending=[True, False, True])
    .drop_duplicates('order_id')
    [['order_id', 'product_category_name_english']]
    .rename(columns={'product_category_name_english': 'main_category'})
)

seller_mode = (
    items_enriched.groupby(['order_id', 'seller_id'])
    .size()
    .reset_index(name='cnt')
    .sort_values(['order_id', 'cnt', 'seller_id'], ascending=[True, False, True])
    .drop_duplicates('order_id')
    [['order_id', 'seller_id']]
    .rename(columns={'seller_id': 'main_seller_id'})
)

In [16]:
seller_mode = seller_mode.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    left_on='main_seller_id',
    right_on='seller_id',
    how='left'
).drop(columns='seller_id')

seller_mode = seller_mode.rename(columns={
    'seller_city': 'main_seller_city',
    'seller_state': 'main_seller_state'
})

In [17]:
items_agg = (
    items_agg
    .merge(category_mode, on='order_id', how='left')
    .merge(seller_mode, on='order_id', how='left')
)

## 14. Creating the Master Dataset

The cleaned and aggregated datasets are combined into a consolidated master dataset using `order_id` and `customer_id` as the relevant joining keys.

The master dataset integrates:

- Order information.
- Customer details.
- Product and category information.
- Seller information.
- Payment metrics.
- Customer review metrics.

The resulting dataset is designed to maintain one record per order, making it suitable for order-level business analysis and Power BI reporting.

In [18]:
master = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(items_agg, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_agg, on='order_id', how='left')
)

## 15. Handling Missing Values

Missing values in selected numerical aggregation columns are replaced with zero, while missing categorical attributes are assigned an `Unknown` label.

This ensures that missing aggregated values do not prevent downstream calculations and visualizations.

Original missing values in date and review-score columns are retained where appropriate, as they represent information that may not be available for certain orders.

In [19]:
fill_zero_cols = [
    'total_item_value', 'total_freight_value', 'total_items', 'unique_products',
    'unique_sellers', 'total_payment_value', 'avg_payment_installments',
    'max_payment_installments', 'payment_type_nunique', 'review_count'
]

for col in fill_zero_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0)

master['main_category'] = master['main_category'].fillna('Unknown')
master['main_payment_type'] = master['main_payment_type'].fillna('Unknown')
master['main_seller_city'] = master['main_seller_city'].fillna('Unknown')
master['main_seller_state'] = master['main_seller_state'].fillna('Unknown')

## 16. Feature Engineering

Additional analytical features are created to support time-based, revenue, and delivery performance analysis.

### Time-Based Features
- Purchase year.
- Purchase month.
- Purchase month name.
- Year-month.
- Purchase day name.

### Delivery Features
- Delivery days: Difference between purchase date and actual customer delivery date.
- Delivery delay days: Difference between actual and estimated delivery dates.
- Delayed flag: Identifies orders delivered after the estimated date.

### Revenue Features
- Order revenue: Total item value, excluding freight.
- Order total cost to customer: Total item value plus freight charges.

In [20]:
master['order_purchase_year'] = master['order_purchase_timestamp'].dt.year
master['order_purchase_month'] = master['order_purchase_timestamp'].dt.month
master['order_purchase_month_name'] = master['order_purchase_timestamp'].dt.month_name()
master['year_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)
master['purchase_day_name'] = master['order_purchase_timestamp'].dt.day_name()

master['delivery_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.days

master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] - master['order_estimated_delivery_date']
).dt.days

master['is_delayed'] = np.where(master['delivery_delay_days'] > 0, 1, 0)
master.loc[master['delivery_delay_days'].isna(), 'is_delayed'] = np.nan

master['order_revenue'] = master['total_item_value']
master['order_total_cost_to_customer'] = master['total_item_value'] + master['total_freight_value']

## 17. Customer Review Score Segmentation

Customer review scores are grouped into three categories to simplify customer satisfaction analysis.

The segments are:

- **Low:** Review scores of 1–2.
- **Medium:** Review score of 3.
- **High:** Review scores of 4–5.

These categories make it easier to compare review distributions across product categories, states, and delivery performance groups.

In [21]:
master['review_bucket'] = pd.cut(
    master['review_score'],
    bins=[0, 2, 3, 5],
    labels=['Low (1-2)', 'Medium (3)', 'High (4-5)'],
    include_lowest=True
)

## 18. Payment Installment Segmentation

Payment installment values are grouped into meaningful ranges to analyze customer payment behavior.

The categories include:

- 1 installment.
- 2–3 installments.
- 4–6 installments.
- 7–12 installments.
- 13 or more installments.

The maximum installment value per order is used for this segmentation.

In [22]:
master['installment_bucket'] = pd.cut(
    master['max_payment_installments'],
    bins=[-1, 1, 3, 6, 12, 100],
    labels=['1', '2-3', '4-6', '7-12', '13+']
)

## 19. Order Value Segmentation

Orders are classified into five value segments based on their total cost to the customer, including item and freight values.

The segments are:

- Very Low.
- Low.
- Medium.
- High.
- Very High.

This classification supports the analysis of order-value distribution and customer purchasing patterns.

The segment boundaries are defined in the feature engineering code.

In [23]:
master['order_value_segment'] = pd.cut(
    master['order_total_cost_to_customer'],
    bins=[-1, 100, 250, 500, 1000, np.inf],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

## 20. Delivery Delay Segmentation

Orders are grouped into delivery-delay categories based on the difference between actual and estimated delivery dates.

The categories are:

- **On Time/Early:** Delivered on or before the estimated date.
- **1–3 Days Late:** Delivered 1–3 days after the estimated date.
- **4–7 Days Late:** Delivered 4–7 days after the estimated date.
- **8+ Days Late:** Delivered 8 or more days after the estimated date.

This segmentation supports the analysis of delivery performance and its relationship with customer satisfaction.

In [24]:
master['delay_band'] = pd.cut(
    master['delivery_delay_days'],
    bins=[-np.inf, 0, 3, 7, np.inf],
    labels=['On Time/Early', '1-3 Days Late', '4-7 Days Late', '8+ Days Late']
)

## 21. Delivered Order Identification

A binary delivery status indicator is created to identify completed deliveries.

- `1`: Order status is delivered.
- `0`: Order status is not delivered.

This feature can be used to distinguish completed orders from orders that are still in progress or have another retained status.

In [25]:
master['is_delivered'] = np.where(master['order_status'] == 'delivered', 1, 0)

## 22. Final Dataset Validation

Final validation checks are performed to assess the structure and quality of the consolidated dataset.

The validation includes:

- Checking the dimensions of the master and delivered-only datasets.
- Examining the percentage of missing values.
- Checking for duplicate order IDs.
- Reviewing selected columns from the final master dataset.
- Confirming that the dataset maintains the intended order-level granularity.

These checks help identify potential data quality issues before exporting the processed dataset.

In [26]:
print("Master shape:", master.shape)
print("\nNull %:\n", (master.isnull().mean() * 100).sort_values(ascending=False).head(15))
print("\nDuplicate order_ids in master:", master['order_id'].duplicated().sum())

master[['order_id', 'year_month', 'order_revenue', 'main_category', 'main_payment_type',
        'review_score', 'delivery_days', 'delivery_delay_days', 'customer_state']].head()

Master shape: (98207, 44)

Null %:
 order_delivered_customer_date    1.768713
delivery_delay_days              1.768713
delay_band                       1.768713
is_delayed                       1.768713
delivery_days                    1.768713
review_bucket                    0.747401
latest_review_date               0.747401
review_score                     0.747401
order_delivered_carrier_date     0.635393
order_approved_at                0.019347
main_seller_id                   0.008146
order_id                         0.000000
customer_state                   0.000000
customer_city                    0.000000
customer_zip_code_prefix         0.000000
dtype: float64

Duplicate order_ids in master: 0


,order_id,year_month,order_revenue,main_category,main_payment_type,review_score,delivery_days,delivery_delay_days,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10,29.99,housewares,voucher,4.0,8.0,-8.0,SP
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07,118.70,perfumery,boleto,4.0,13.0,-6.0,BA
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08,159.90,auto,credit_card,5.0,9.0,-18.0,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11,45.00,pet_shop,credit_card,5.0,13.0,-13.0,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02,19.90,stationery,credit_card,5.0,2.0,-10.0,SP


## 23. Exporting the Final Dataset

The processed master dataset is exported to CSV format for further analysis and visualization.

**Output file:** `Master_ecommerce.csv`

The exported dataset serves as the primary input for the Power BI Brazilian E-Commerce Analytics Dashboard.

In [27]:
master.to_csv('../Final_data/Master_ecommerce.csv', index=False)

# 24. Conclusion

The data preparation process transformed multiple raw Brazilian E-Commerce datasets into a consolidated order-level dataset.

The workflow included data cleaning, standardization, aggregation, dataset integration, feature engineering, and final validation.

The resulting dataset provides a foundation for analyzing:

- Revenue and order performance.
- Product category and seller performance.
- Customer purchasing behavior.
- Payment methods and installment patterns.
- Delivery performance.
- Customer review scores and satisfaction.

The final dataset was exported for use in Power BI, where interactive dashboards were developed to support business performance analysis and decision-making.